## EXP-INGEST-003 — AI Metadata Enrichment

### Purpose

After deterministic extraction, I want CEREBRO to use AI to suggest useful metadata that cannot be reliably obtained from the file alone.

AI suggestions remain separate from source facts. They are presented to the user for review and can be edited, accepted, or rejected before the artifact enters the Digital Knowledge Twin.

**Principle:** AI suggests. The user confirms.

### 1. Make 02C independent

In [43]:
from pathlib import Path
import json

repo_root = Path.cwd().parents[1]

source_path = (
    repo_root
    / "poc/data/raw/text/benchmark_001.txt"
)

benchmark_path = (
    repo_root
    / "poc/data/ground_truth/CEREBRO-GT-v0.1.json"
)

assert source_path.exists()
assert benchmark_path.exists()

source_text = source_path.read_text(encoding="utf-8")

with benchmark_path.open("r", encoding="utf-8") as f:
    benchmark = json.load(f)

ground_truth_annotation = benchmark["annotations"][0]

print("✓ Source loaded:", source_path.name)
print("✓ Benchmark loaded:", benchmark["benchmark_id"])

✓ Source loaded: benchmark_001.txt
✓ Benchmark loaded: CEREBRO-GT-v0.1


### 2. Define the AI enrichment contract

In [44]:
ai_enrichment_schema = {
    "title": None,
    "artifact_type": None,
    "language": None,
    "description": None,

    "authors": [],
    "people": [],
    "organizations": [],
    "projects": [],
    "topics": [],
    "tags": []
}

ai_enrichment_schema

{'title': None,
 'artifact_type': None,
 'language': None,
 'description': None,
 'authors': [],
 'people': [],
 'organizations': [],
 'projects': [],
 'topics': [],
 'tags': []}

### 3. AI instruction to Enrichment Task Contract

In [45]:
# ---------------------------------------------------------
# CEREBRO AI Enrichment Task Contract
# ---------------------------------------------------------

enrichment_task = {
    "task_id": "TASK-ENRICH-001",
    "task_type": "artifact_metadata_enrichment",

    # What the selected model must do
    "instruction": """
Analyze the supplied artifact content and suggest metadata
for the CEREBRO artifact intake form.

Rules:
1. Use only information supported by the artifact.
2. Do not invent missing information.
3. Use null or [] when evidence is insufficient.
4. Do not assume the artifact owner is the author.
5. Preserve names and entities exactly where possible.
6. Topics and tags must be grounded in the content.
7. Keep the description concise.
8. Distinguish explicit information from inference.
9. Return structured metadata using the required schema.
""".strip(),

    # Expected output
    "output_schema": ai_enrichment_schema,

    # Requirements used by the Router Agent
    "requirements": {
        "structured_output": True,
        "grounded_only": True,
        "entity_preservation": True,
        "allow_inference": True
    },

    # Routing policy
    "routing": {
        "strategy": "router_agent",

        # Prefer cheaper/local processing where appropriate
        "prefer_local": True,

        # Router may use a frontier model when necessary
        "frontier_allowed": True,

        # Local output may be escalated when quality is insufficient
        "escalation_allowed": True,

        "decision_factors": [
            "privacy",
            "task_complexity",
            "quality_requirement",
            "confidence",
            "cost",
            "latency"
        ]
    },

    # No model has been selected yet
    "routing_decision": {
        "route": None,
        "model": None,
        "reason": None,
        "escalated": False
    }
}

enrichment_task

{'task_id': 'TASK-ENRICH-001',
 'task_type': 'artifact_metadata_enrichment',
 'instruction': 'Analyze the supplied artifact content and suggest metadata\nfor the CEREBRO artifact intake form.\n\nRules:\n1. Use only information supported by the artifact.\n2. Do not invent missing information.\n3. Use null or [] when evidence is insufficient.\n4. Do not assume the artifact owner is the author.\n5. Preserve names and entities exactly where possible.\n6. Topics and tags must be grounded in the content.\n7. Keep the description concise.\n8. Distinguish explicit information from inference.\n9. Return structured metadata using the required schema.',
 'output_schema': {'title': None,
  'artifact_type': None,
  'language': None,
  'description': None,
  'authors': [],
  'people': [],
  'organizations': [],
  'projects': [],
  'topics': [],
  'tags': []},
 'requirements': {'structured_output': True,
  'grounded_only': True,
  'entity_preservation': True,
  'allow_inference': True},
 'routing': {

In [46]:
assert enrichment_task["routing"]["strategy"] == "router_agent"
assert enrichment_task["routing"]["prefer_local"] is True
assert enrichment_task["routing"]["frontier_allowed"] is True
assert enrichment_task["routing"]["escalation_allowed"] is True

assert enrichment_task["routing_decision"]["model"] is None

print("✓ Enrichment task contract created")
print("✓ Router Agent controls model selection")
print("✓ Local/cheaper model preferred where appropriate")
print("✓ Frontier escalation permitted")
print("✓ No model hard-wired into the pipeline")

✓ Enrichment task contract created
✓ Router Agent controls model selection
✓ Local/cheaper model preferred where appropriate
✓ Frontier escalation permitted
✓ No model hard-wired into the pipeline


### 4. Baseline enrichment

In [47]:
ai_result = {
    "title": {
        "value": "CEREBRO Digital Knowledge Twin",
        "evidence": "inferred_from_content"
    },

    "artifact_type": {
        "value": "knowledge_description",
        "evidence": "inferred_from_content"
    },

    "language": {
        "value": "English",
        "evidence": "detected_from_content"
    },

    "description": {
        "value": (
            "CEREBRO is a Digital Knowledge Twin that preserves "
            "and connects human knowledge while maintaining "
            "provenance to original source artifacts."
        ),
        "evidence": "generated_from_content"
    },

    "authors": {
        "value": [],
        "evidence": "insufficient_evidence"
    },

    "people": {
        "value": [],
        "evidence": "insufficient_evidence"
    },

    "organizations": {
        "value": [],
        "evidence": "insufficient_evidence"
    },

    "projects": {
        "value": [],
        "evidence": "insufficient_evidence"
    },

    "topics": {
        "value": [
            "Digital Knowledge Twin",
            "Provenance",
            "Knowledge Fragment",
            "Source Artifact",
            "Assisted Recollection"
        ],
        "evidence": "extracted_from_content"
    },

    "tags": {
        "value": [
            "CEREBRO",
            "knowledge",
            "provenance",
            "recollection"
        ],
        "evidence": "suggested_from_content"
    }
}

ai_result

{'title': {'value': 'CEREBRO Digital Knowledge Twin',
  'evidence': 'inferred_from_content'},
 'artifact_type': {'value': 'knowledge_description',
  'evidence': 'inferred_from_content'},
 'language': {'value': 'English', 'evidence': 'detected_from_content'},
 'description': {'value': 'CEREBRO is a Digital Knowledge Twin that preserves and connects human knowledge while maintaining provenance to original source artifacts.',
  'evidence': 'generated_from_content'},
 'authors': {'value': [], 'evidence': 'insufficient_evidence'},
 'people': {'value': [], 'evidence': 'insufficient_evidence'},
 'organizations': {'value': [], 'evidence': 'insufficient_evidence'},
 'projects': {'value': [], 'evidence': 'insufficient_evidence'},
 'topics': {'value': ['Digital Knowledge Twin',
   'Provenance',
   'Knowledge Fragment',
   'Source Artifact',
   'Assisted Recollection'],
  'evidence': 'extracted_from_content'},
 'tags': {'value': ['CEREBRO', 'knowledge', 'provenance', 'recollection'],
  'evidence':

### 5. Wrap every suggestion with provenance

In [48]:
# ---------------------------------------------------------
# CEREBRO AI-Suggested Metadata with Provenance
# ---------------------------------------------------------

ai_prefill = {}

for field, result in ai_result.items():

    ai_prefill[field] = {
        "value": result["value"],

        # Classification of this metadata
        "source": "ai_suggested",

        # How the value is supported by the artifact
        "evidence": result["evidence"],

        # Human-in-the-loop state
        "status": "suggested",
        "user_confirmed": False,

        # AI processing provenance
        "provenance": {
            "task_id": enrichment_task["task_id"],
            "task_type": enrichment_task["task_type"],

            # 02C uses a controlled mock baseline.
            # Actual routing begins in 02D.
            "router": "cerebro_router",
            "route": "mock_baseline",
            "model": None,
            "escalated": False
        }
    }


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

for field, metadata in ai_prefill.items():

    assert metadata["source"] == "ai_suggested"
    assert metadata["status"] == "suggested"
    assert metadata["user_confirmed"] is False

    assert (
        metadata["provenance"]["task_id"]
        == enrichment_task["task_id"]
    )

    assert (
        metadata["provenance"]["route"]
        == "mock_baseline"
    )

    assert metadata["provenance"]["model"] is None


print("✓ AI suggestions created")
print("✓ Field-level provenance attached")
print("✓ Suggestions remain unconfirmed")
print("✓ Mock baseline identified")
print("✓ No model falsely attributed")

✓ AI suggestions created
✓ Field-level provenance attached
✓ Suggestions remain unconfirmed
✓ Mock baseline identified
✓ No model falsely attributed


### 6. Show the future prefilled form

In [49]:
print("CEREBRO — Artifact Metadata")
print("---------------------------")

for field, metadata in ai_prefill.items():

    value = metadata["value"]

    if value in [None, []]:
        display_value = "[Not determined]"
    else:
        display_value = value

    print(
        f"{field:15}: {display_value}"
    )

CEREBRO — Artifact Metadata
---------------------------
title          : CEREBRO Digital Knowledge Twin
artifact_type  : knowledge_description
language       : English
description    : CEREBRO is a Digital Knowledge Twin that preserves and connects human knowledge while maintaining provenance to original source artifacts.
authors        : [Not determined]
people         : [Not determined]
organizations  : [Not determined]
projects       : [Not determined]
topics         : ['Digital Knowledge Twin', 'Provenance', 'Knowledge Fragment', 'Source Artifact', 'Assisted Recollection']
tags           : ['CEREBRO', 'knowledge', 'provenance', 'recollection']


### 7. Evaluate topics against ground truth

In [50]:
expected_topics = {
    concept["label"]
    for concept
    in ground_truth_annotation["concepts"]
}

predicted_topics = set(
    ai_result["topics"]["value"]
)

true_positive = (
    expected_topics & predicted_topics
)

false_positive = (
    predicted_topics - expected_topics
)

false_negative = (
    expected_topics - predicted_topics
)

precision = (
    len(true_positive) / len(predicted_topics)
    if predicted_topics else 0
)

recall = (
    len(true_positive) / len(expected_topics)
    if expected_topics else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("Expected :", expected_topics)
print("Predicted:", predicted_topics)

print("\nTrue Positive :", true_positive)
print("False Positive:", false_positive)
print("False Negative:", false_negative)

print("\nPrecision:", round(precision, 3))
print("Recall   :", round(recall, 3))
print("F1       :", round(f1, 3))

Expected : {'Provenance', 'Digital Knowledge Twin', 'Knowledge Fragment', 'Assisted Recollection', 'Source Artifact'}
Predicted: {'Provenance', 'Digital Knowledge Twin', 'Knowledge Fragment', 'Assisted Recollection', 'Source Artifact'}

True Positive : {'Provenance', 'Digital Knowledge Twin', 'Knowledge Fragment', 'Assisted Recollection', 'Source Artifact'}
False Positive: set()
False Negative: set()

Precision: 1.0
Recall   : 1.0
F1       : 1.0


### 8. Hallucination check

In [51]:
unsupported_identity_claims = []

for field in [
    "authors",
    "people",
    "organizations",
    "projects"
]:
    values = ai_result[field]["value"]

    if values:
        unsupported_identity_claims.extend(
            [
                {
                    "field": field,
                    "value": value
                }
                for value in values
            ]
        )

print(
    "Unsupported identity claims:",
    unsupported_identity_claims
)

assert len(unsupported_identity_claims) == 0

print("✓ No unsupported identity claims")

Unsupported identity claims: []
✓ No unsupported identity claims


### 9. Intake state

In [52]:
intake_state = {
    "artifact_status": "staged",

    "deterministic_extraction": "complete",

    "ai_enrichment": {
        "status": "baseline_complete",

        "routing": {
            "router": "cerebro_router",
            "route": "mock_baseline",
            "model": None,
            "escalated": False
        },

        "user_confirmed": False
    },

    "user_review": "pending",
    "registration": "pending",
    "knowledge_ingestion": "blocked"
}

intake_state

{'artifact_status': 'staged',
 'deterministic_extraction': 'complete',
 'ai_enrichment': {'status': 'baseline_complete',
  'routing': {'router': 'cerebro_router',
   'route': 'mock_baseline',
   'model': None,
   'escalated': False},
  'user_confirmed': False},
 'user_review': 'pending',
 'registration': 'pending',
 'knowledge_ingestion': 'blocked'}

### 10. Experiment result

In [53]:
print("EXP-INGEST-003")
print("----------------")

print("AI contract       : ✓")
print("Field provenance  : ✓")
print("Topic evaluation  : ✓")
print("Hallucination test: ✓")
print("Human review      : PENDING")
print("Artifact status   : STAGED")

print("\nEXP-INGEST-003 BASELINE: PASS")

EXP-INGEST-003
----------------
AI contract       : ✓
Field provenance  : ✓
Topic evaluation  : ✓
Hallucination test: ✓
Human review      : PENDING
Artifact status   : STAGED

EXP-INGEST-003 BASELINE: PASS


### Result

CEREBRO now has a model-independent contract for AI-assisted artifact enrichment.

Metadata suggestions remain separate from deterministic facts and can be evaluated against trusted ground truth before user confirmation.

Model selection is not hard-wired into the enrichment pipeline. A Router Agent will decide whether each enrichment task should use a local/cheaper model or escalate to a frontier model based on privacy, complexity, quality requirements, latency, and cost.

The next experiment will implement and evaluate this routing decision.

flowchart LR
    A[Artifact] --> D[Deterministic Facts]
    D --> AI[AI Enrichment]
    AI --> S[Suggested Metadata]
    S --> E[Evaluate]
    E --> H[Human Review]
    H -->|Edit / Accept| SUB[Submit]
    SUB --> R[Register Artifact]
    R --> K[Knowledge Ingestion]